# Single External Image — Human Evaluation

Upload **one external image**, generate a caption with the fine-tuned Al-Balad Qwen3-VL-2B model, then record a human evaluation.

This notebook is for **qualitative human evaluation**, not the formal held-out benchmark.

## 1. Install packages

In [ ]:
!pip install -q transformers==5.14.1 peft==0.19.1 accelerate==1.14.0 bitsandbytes==0.49.2 pillow-heif==1.1.1 ipywidgets

## 2. Imports and Google Drive

In [ ]:
import gc, json
from pathlib import Path
import pandas as pd
import torch
import ipywidgets as widgets
from PIL import Image, ImageOps, ImageStat
from IPython.display import display, Markdown
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import PeftModel
from pillow_heif import register_heif_opener
from google.colab import drive, files

register_heif_opener()
drive.mount('/content/drive')

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Colab first.')

print('GPU:', torch.cuda.get_device_name(0))

## 3. Load experiment settings

In [ ]:
EXPERIMENT_NAME = 'qwen3_vl_2b_albalad_v1'
RUN_ROOT = Path('/content/drive/MyDrive/albalad_runs') / EXPERIMENT_NAME
FINAL_ADAPTER_DIR = RUN_ROOT / 'final_lora_adapter'
PROCESSOR_DIR = RUN_ROOT / 'processor'
EXPERIMENT_CONFIG_PATH = RUN_ROOT / 'experiment_config.json'

if not FINAL_ADAPTER_DIR.exists():
    raise FileNotFoundError(FINAL_ADAPTER_DIR)
if not PROCESSOR_DIR.exists():
    raise FileNotFoundError(PROCESSOR_DIR)
if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(EXPERIMENT_CONFIG_PATH)

with EXPERIMENT_CONFIG_PATH.open('r', encoding='utf-8') as f:
    config = json.load(f)

BASE_MODEL_ID = config['base_model_id']
TARGET_WIDTH = int(config['image']['prepared_width'])
TARGET_HEIGHT = int(config['image']['prepared_height'])

print('Model:', BASE_MODEL_ID)
print('Image size:', TARGET_WIDTH, 'x', TARGET_HEIGHT)

## 4. Upload exactly one image

In [ ]:
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError('Please upload exactly ONE image.')

filename = next(iter(uploaded))
image_path = Path('/content') / filename
image_path.write_bytes(uploaded[filename])
print('Uploaded:', image_path)

## 5. Preprocess and display the image

The image is converted to RGB, resized while preserving aspect ratio, and padded to the same size used during training.

If automatic day/night detection is wrong, set `LIGHTING_OVERRIDE` to `'day'` or `'night'`.

In [ ]:
LIGHTING_OVERRIDE = None   # None, 'day', or 'night'

with Image.open(image_path) as im:
    im = ImageOps.exif_transpose(im).convert('RGB')
    original_size = im.size
    contained = ImageOps.contain(im, (TARGET_WIDTH, TARGET_HEIGHT), method=Image.Resampling.LANCZOS)

processed_image = Image.new('RGB', (TARGET_WIDTH, TARGET_HEIGHT), (0, 0, 0))
x = (TARGET_WIDTH - contained.width) // 2
y = (TARGET_HEIGHT - contained.height) // 2
processed_image.paste(contained, (x, y))

mean_luminance = float(ImageStat.Stat(processed_image.convert('L')).mean[0])

if LIGHTING_OVERRIDE in {'day', 'night'}:
    lighting = LIGHTING_OVERRIDE
else:
    lighting = 'day' if mean_luminance >= 70 else 'night'

PROMPT = (
    'Describe this historic building in Al-Balad during the day.'
    if lighting == 'day'
    else 'Describe this historic building in Al-Balad during the night.'
)

display(processed_image)
print('Original size:', original_size)
print('Processed size:', processed_image.size)
print('Lighting:', lighting)
print('Prompt:', PROMPT)

## 6. Load the fine-tuned model

In [ ]:
processor = AutoProcessor.from_pretrained(str(PROCESSOR_DIR))
processor.tokenizer.padding_side = 'left'
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

USE_BF16 = bool(torch.cuda.is_bf16_supported())
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=COMPUTE_DTYPE,
    attn_implementation='sdpa',
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(
    base_model,
    str(FINAL_ADAPTER_DIR),
    is_trainable=False,
)
model.eval()
model.config.use_cache = True
print('Fine-tuned model loaded.')

## 7. Generate caption

In [ ]:
messages = [{
    'role': 'user',
    'content': [
        {'type': 'image'},
        {'type': 'text', 'text': PROMPT},
    ],
}]

prompt_text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(
    text=[prompt_text],
    images=[processed_image],
    return_tensors='pt',
)
inputs.pop('token_type_ids', None)

device = next(p.device for p in model.parameters() if p.device.type != 'meta')
inputs = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in inputs.items()}
input_length = inputs['input_ids'].shape[1]

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        num_beams=1,
        repetition_penalty=1.05,
        use_cache=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

generated_only = generated_ids[:, input_length:]
caption = processor.batch_decode(
    generated_only,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0].strip()

display(Markdown('## Generated Caption\n\n' + caption))

## 8. Human evaluation form

In [ ]:
building_identity = widgets.Dropdown(
    options=[
        ('Not evaluated', ''),
        ('Correct', '1'),
        ('Incorrect', '0'),
        ('Unknown / Not applicable', 'NA'),
    ],
    description='Identity:',
)

architectural_correctness = widgets.IntSlider(value=3, min=1, max=5, description='Architecture:')
visual_grounding = widgets.IntSlider(value=3, min=1, max=5, description='Grounding:')
historical_factuality = widgets.Dropdown(
    options=[
        ('Not evaluated', ''),
        ('1 - Very poor', '1'),
        ('2 - Poor', '2'),
        ('3 - Moderate', '3'),
        ('4 - Good', '4'),
        ('5 - Excellent', '5'),
        ('Unknown / NA', 'NA'),
    ],
    description='History:',
)
hallucination = widgets.IntSlider(value=1, min=1, max=5, description='Hallucination:')
fluency = widgets.IntSlider(value=3, min=1, max=5, description='Fluency:')
notes = widgets.Textarea(description='Notes:', placeholder='Optional evaluator notes...', layout=widgets.Layout(width='80%'))

display(building_identity, architectural_correctness, visual_grounding, historical_factuality, hallucination, fluency, notes)

print('Architecture / Grounding / History / Fluency: 1 = very poor, 5 = excellent')
print('Hallucination: 1 = none, 5 = severe')

## 9. Save and download the evaluation

In [ ]:
result = pd.DataFrame([{
    'image': filename,
    'lighting': lighting,
    'prompt': PROMPT,
    'generated_caption': caption,
    'building_identity': building_identity.value,
    'architectural_correctness_1_5': architectural_correctness.value,
    'visual_grounding_1_5': visual_grounding.value,
    'historical_factuality_1_5_or_NA': historical_factuality.value,
    'hallucination_severity_1_5': hallucination.value,
    'fluency_1_5': fluency.value,
    'notes': notes.value,
}])

display(result)

OUTPUT_PATH = Path('/content/single_image_human_evaluation.csv')
result.to_csv(OUTPUT_PATH, index=False)
files.download(str(OUTPUT_PATH))

## 10. Optional — release GPU memory

In [ ]:
del model
del base_model
gc.collect()
torch.cuda.empty_cache()
print('GPU memory released.')